## List Indexes

In [4]:
!curl -X GET "http://localhost:9200/_cat/indices?v"


health status index                        uuid                   pri rep docs.count docs.deleted store.size pri.store.size
green  open   top_queries-2025.07.09-40803 qicDITvbSiiR2B9MewaqOg   1   0          6           12     70.5kb         70.5kb
green  open   .plugins-ml-config           Xw2e5viYTBW40wCGaCq20Q   1   0          1            0      3.9kb          3.9kb
yellow open   gds-i-person                 4k0dDwlEQs2kDKe_nRP84w   1   1          0            0       208b           208b
yellow open   people                       6UmC9f2nTLexXtgyuQuzcA   1   1         11            0       13kb           13kb


## Count of records by index

In [17]:
!curl -X GET "http://localhost:9200/people/_count"

{"count":1000,"_shards":{"total":1,"successful":1,"skipped":0,"failed":0}}

## Find  first record

In [19]:
## Retrieve record with id "person-0"

import subprocess
import json

# Query to find the record with id "person-0"
query = {
    "query": {
        "term": {
            "id": "person-0"
        }
    },
    "_source": True
}

result = subprocess.run([
    'curl', '-X', 'POST', 
    'http://localhost:9200/people/_search',
    '-H', 'Content-Type: application/json',
    '-d', json.dumps(query)
], capture_output=True, text=True)

# Parse and display the response
try:
    response_json = json.loads(result.stdout)
    
    if 'hits' in response_json and 'hits' in response_json['hits']:
        hits = response_json['hits']['hits']
        if hits:
            print(f"Found {len(hits)} record(s) with id 'person-0':")
            print("=" * 50)
            
            for i, hit in enumerate(hits, 1):
                print(f"\nRecord {i}:")
                print(json.dumps(hit['_source'], indent=2))
                print("-" * 30)
        else:
            print("No records found with id 'person-0'")
            print("Response:", json.dumps(response_json, indent=2))
    else:
        print("Unexpected response format:")
        print(json.dumps(response_json, indent=2))
        
except json.JSONDecodeError:
    print("Raw response:")
    print(result.stdout)
    if result.stderr:
        print("Error:")
        print(result.stderr)

No records found with id 'person-0'
Response: {
  "took": 2,
  "timed_out": false,
  "_shards": {
    "total": 1,
    "successful": 1,
    "skipped": 0,
    "failed": 0
  },
  "hits": {
    "total": {
      "value": 0,
      "relation": "eq"
    },
    "max_score": null,
    "hits": []
  }
}


## Find by partial street

In [16]:
!curl -X POST "http://localhost:9200/gds-i-person/_search" -H "Content-Type: application/json" -d '{"query":{"match_phrase":{"addresses.street":"Robert"}},"_source":true}'

{"took":3,"timed_out":false,"_shards":{"total":1,"successful":1,"skipped":0,"failed":0},"hits":{"total":{"value":0,"relation":"eq"},"max_score":null,"hits":[]}}

In [10]:
!curl -X POST "http://localhost:9200/gds-i-person/_search" -H "Content-Type: application/json" -d '{"query":{"nested":{"path":"addresses","query":{"match_phrase":{"addresses.street":"67318 Robert"}}}}}'



{"took":11,"timed_out":false,"_shards":{"total":1,"successful":1,"skipped":0,"failed":0},"hits":{"total":{"value":0,"relation":"eq"},"max_score":null,"hits":[]}}

In [ ]:
!curl -X POST "http://localhost:9200/gds-i-person/_search" -H "Content-Type: application/json" -d '{"query":{"nested":{"path":"addresses","query":{"match_phrase":{"addresses.street":"67318 Robert"}}}}}'



In [15]:
!curl -X POST "http://localhost:9200/gds-i-person/_search" -H "Content-Type: application/json" -   !curl -X GET "http://localhost:9200/gds-i-person/_mapping?pretty"

curl: option -: is unknown
curl: try 'curl --help' or 'curl --manual' for more information


In [20]:
curl -X POST "http://localhost:8200/gds-i-person/_search" \
-H "Content-Type: application/json" \
-d '{
  "query": {
    "nested": {
      "path": "addresses",
      "query": {
        "match_phrase": {
          "addresses.street": "2025 Rachel"
        }
      }
    }
  }
}'

SyntaxError: unterminated string literal (detected at line 3) (2961286920.py, line 3)

In [24]:
import subprocess
import json

query = {
    "query": {
        "nested": {
            "path": "addresses",
            "query": {
                "match_phrase": {
                    "addresses.street": "2025 Rachel"
                }
            }
        }
    }
}

result = subprocess.run([
    'curl', '-X', 'POST', 
    'http://localhost:8200/people/_search',
    '-H', 'Content-Type: application/json',
    '-d', json.dumps(query)
], capture_output=True, text=True)

# Parse and pretty-print the JSON response
try:
    response_json = json.loads(result.stdout)
    print(json.dumps(response_json, indent=2))
except json.JSONDecodeError:
    # If response is not valid JSON, print raw output
    print("Raw response:")
    print(result.stdout)
    if result.stderr:
        print("Error:")
        print(result.stderr)

{
  "error": {
    "root_cause": [
      {
        "type": "query_shard_exception",
        "reason": "failed to create query: [nested] nested object under path [addresses] is not of nested type",
        "index": "people",
        "index_uuid": "SJ2xxn-EQJeIvDVFKgd0bg"
      }
    ],
    "type": "search_phase_execution_exception",
    "reason": "all shards failed",
    "phase": "query",
    "grouped": true,
    "failed_shards": [
      {
        "shard": 0,
        "index": "people",
        "node": "6EwnINmST0W1fYfCRE83Mw",
        "reason": {
          "type": "query_shard_exception",
          "reason": "failed to create query: [nested] nested object under path [addresses] is not of nested type",
          "index": "people",
          "index_uuid": "SJ2xxn-EQJeIvDVFKgd0bg",
          "caused_by": {
            "type": "illegal_state_exception",
            "reason": "[nested] nested object under path [addresses] is not of nested type"
          }
        }
      }
    ]
  },
  "sta

In [23]:
import subprocess
import json

query = {
    "query": {
        "nested": {
            "path": "addresses",
            "query": {
                "match_phrase": {
                    "addresses.street": "2025 Rachel"
                }
            }
        }
    }
}

result = subprocess.run([
    'curl', '-X', 'POST', 
    'http://localhost:8200/gds-i-person/_search',
    '-H', 'Content-Type: application/json',
    '-d', json.dumps(query)
], capture_output=True, text=True)

# Parse and extract person records
try:
    response_json = json.loads(result.stdout)
    
    if 'hits' in response_json and 'hits' in response_json['hits']:
        print(f"Found {response_json['hits']['total']['value']} person record(s):")
        print("=" * 50)
        
        for i, hit in enumerate(response_json['hits']['hits'], 1):
            print(f"\nPerson Record {i}:")
            print(json.dumps(hit['_source'], indent=2))
            print("-" * 30)
    else:
        print("No person records found")
        
except json.JSONDecodeError:
    print("Raw response:")
    print(result.stdout)
    if result.stderr:
        print("Error:")

Found 0 person record(s):
